In [1]:
import numpy as np
import pandas as pd

In [3]:
pip install kiteconnect

In [ ]:
pip install pyotp

In [2]:
from zerodha import Zerodha
from kiteconnect import KiteConnect

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")


# ---------------- KITE LOGIN ----------------
kite = Zerodha(user_id='AI2416', password='Vinay@14541228', twofa='6TWJRKAU6TYJ3O2U7EYAIJEKPS3BDQBD')
kite.login()

# ---------------------- USER INPUT ----------------------
equity_details = pd.read_csv("ind_nifty50list.csv")
tickers = equity_details['Symbol'].tolist()
#tickers = ['RELIANCE', 'TCS', 'HDFCBANK']   # removed .NS
# tickers = [
#  'ADANIENT.NS','ADANIPORTS.NS','APOLLOHOSP.NS','ASIANPAINT.NS','AXISBANK.NS','BAJAJ-AUTO.NS',
#  'BAJFINANCE.NS','BAJAJFINSV.NS','BEL.NS','BHARTIARTL.NS','CIPLA.NS','COALINDIA.NS','DRREDDY.NS',
#  'EICHERMOT.NS','ETERNAL.NS','GRASIM.NS','HCLTECH.NS','HDFCBANK.NS','HDFCLIFE.NS','HINDALCO.NS',
#  'HINDUNILVR.NS','ICICIBANK.NS','ITC.NS','INFY.NS','INDIGO.NS','JSWSTEEL.NS','JIOFIN.NS',
#  'KOTAKBANK.NS','LT.NS','M&M.NS','MARUTI.NS','MAXHEALTH.NS','NTPC.NS','NESTLEIND.NS','ONGC.NS',
#  'POWERGRID.NS','RELIANCE.NS','SBILIFE.NS','SHRIRAMFIN.NS','SBIN.NS','SUNPHARMA.NS','TCS.NS',
#  'TATACONSUM.NS','TMPV.NS','TATASTEEL.NS','TECHM.NS','TITAN.NS','TRENT.NS','ULTRACEMCO.NS','WIPRO.NS'
# ]
signal_start = pd.to_datetime("2023-09-01").tz_localize(None)
signal_end   = pd.to_datetime("2025-11-30").tz_localize(None)


# Start date = 2.5 years (30 months) BEFORE signal_start
start_date = signal_start - pd.DateOffset(months=30)

# End date = 10 days AFTER signal_end
end_date = signal_end + pd.Timedelta(days=10)

train_window = 504
rv_window = 10

out_csv = "master_signals.csv"

# --------------------------------------------------------

instruments = kite.instruments("NSE")

def get_instrument_token(symbol):
    return next((inst['instrument_token'] for inst in instruments if inst['tradingsymbol'] == symbol), None)


def fetch_kite_data(symbol, start_date, end_date, interval="day"):

    try:
        token = get_instrument_token(symbol)

        if token is None:
            print(f"No instrument found for {symbol}")
            return None

        data = kite.historical_data(token, start_date.strftime('%Y-%m-%d'),
                                    end_date.strftime('%Y-%m-%d'),
                                    interval)

        if not data:
            return None

        df = pd.DataFrame(data)
        df['Date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
        df = df.rename(columns={
            "open":"Open",
            "high":"High",
            "low":"Low",
            "close":"Close",
            "volume":"Volume"
        })

        df = df[['Date','Open','High','Low','Close','Volume']]
        return df

    except Exception as e:
        print(f"Error fetching {symbol} -> {e}")
        return None



# ----------------- DOWNLOAD NIFTY ONCE -----------------

print("\nDownloading NIFTY using Kite...")

nifty_token = next((i['instrument_token'] for i in instruments if i['tradingsymbol'] == "NIFTY 50"), None)

nifty_data = kite.historical_data(
    nifty_token,
    start_date.strftime('%Y-%m-%d'),
    end_date.strftime('%Y-%m-%d'),
    "day"
)

nifty = pd.DataFrame(nifty_data)
# nifty['Date'] = pd.to_datetime(nifty['date'])
nifty['Date'] = pd.to_datetime(nifty['date']).dt.tz_localize(None)

nifty = nifty[['Date','close']].rename(columns={'close':'Index_Close'})


# ----------------- FUNCTIONS -----------------

def get_alpha_beta(stock_returns, index_returns):
    df = pd.concat([stock_returns, index_returns], axis=1).dropna()
    df.columns = ['stock', 'index']

    if len(df) < 50:
        return None, None

    cov = np.cov(df['stock'], df['index'], ddof=1)[0, 1]
    var = np.var(df['index'], ddof=1)

    if var == 0:
        return None, None

    beta  = cov / var
    alpha = df['stock'].mean() - beta * df['index'].mean()

    return float(alpha), float(beta)



def arimax_signal_from_history(history_df):

    df = history_df.copy()

    df['log_return']  = np.log(df['Close'] / df['Close'].shift(1))
    df['RV']           = df['log_return'].rolling(rv_window).std() * np.sqrt(252)
    df['Volume_Diff']  = df['Volume'].pct_change()
    df['Index_Return'] = np.log(df['Index_Close'] / df['Index_Close'].shift(1))

    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    if df.empty or len(df) < 80:
        return None, None, None, None

    alpha, beta = get_alpha_beta(df['log_return'], df['Index_Return'])
    if alpha is None or beta is None:
        return None, None, None, None

    y = df['Close']

    X = df[['RV', 'Volume_Diff']].copy()
    X['Alpha'] = alpha
    X['Beta']  = beta

    try:
        model = SARIMAX(
            y,
            exog = X,
            order = (1,1,1),
            enforce_stationarity = False,
            enforce_invertibility = False
        )

        res = model.fit(disp=False)

        last_exog = X.iloc[-1].values.reshape(1, -1)

        fc = res.get_forecast(steps=1, exog=last_exog)

        next_price = float(fc.predicted_mean.iloc[0])
        last_close = float(y.iloc[-1])

        signal = 1 if next_price > last_close else 0

        last_rv = float(df['RV'].iloc[-1])

        return int(signal), last_rv, alpha, beta

    except:
        return None, None, None, None


# ----------------- MAIN LOOP -----------------

master_rows = []

for ticker in tqdm(tickers, desc="Tickers"):

    df = fetch_kite_data(ticker, start_date, end_date)

    if df is None or df.empty:
        continue

    df['Date'] = pd.to_datetime(df['Date'])

    df = df.merge(nifty, on='Date', how='left')
    df['Index_Close'] = df['Index_Close'].ffill()

    df = df.sort_values('Date').reset_index(drop=True)

    # ------ CREATE SIGNALS ------
    for i in range(train_window, len(df)):

        current_date = df.iloc[i]['Date']

        if not (signal_start <= current_date <= signal_end):
            continue

        history = df.iloc[i - train_window : i].copy().set_index('Date')

        signal, last_rv, alpha, beta = arimax_signal_from_history(history)

        if signal is None:
            continue

        today_row = df.iloc[i]

        master_rows.append({
            "Date": current_date.date(),
            "Ticker": ticker,

            "Open": float(today_row['Open']),
            "High": float(today_row['High']),
            "Low": float(today_row['Low']),
            "Close": float(today_row['Close']),
            "Adj Close": float(today_row['Close']),
            "Volume": float(today_row['Volume']),
            "Index_Close": float(today_row['Index_Close']),

            "Signal": int(signal),
            "RV": float(last_rv),
            "Alpha": float(alpha),
            "Beta": float(beta)
        })


# ---------------- FINAL OUTPUT ----------------

master_df = pd.DataFrame(master_rows)
master_df = master_df.sort_values(['Date','Ticker']).reset_index(drop=True)

print("\n✅ Generated signals:", len(master_df))
print(master_df.head())

master_df.to_csv("master_df_8.csv", index=False)
# print("\n✅ Saved as master_df_4.csv")
